# PROJ951 · Notebook 0 · Python introduction

<table>
  <tr>
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/PROJ951/main/notebooks/logos/solaracademy.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/PROJ951/main/notebooks/logos/usmb.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/PROJ951/main/notebooks/logos/cnrs.png" width="80" />
  </tr>
</table>

<sub>Author: Mohamed Ennhiri, PhD student, LOCIE UMR CNRS 5271, Université Savoie Mont Blanc</sub>

<sub>Data: PVGIS 5.3 (SARAH3 and ERA5) for the production, synthetic profiles for the demand of the three buildings.</sub>

**Goal.** Learn the pieces of Python you meet in the TD notebooks and in TP1, on the numbers of the course. The formulas come from the lectures. This notebook teaches the syntax only.

**For everyone.** SoLEM students: read each line, run the cell, check the number. ESBC students: go faster and help your pair. Both jobs count in TP1.

**How a notebook works.**
- A notebook mixes text cells and code cells. Click a code cell and press **Shift + Enter** to run it. The result appears under the cell.
- Run the cells in order, from the top. A cell uses the names created by the cells above it.
- In each exercise, replace every `...` and every `# TODO`, then run the check under the exercise.
- First action on Google Colab: **File, Save a copy in Drive**. Without a copy your work disappears when the tab closes.

**Time.** About 45 minutes, in CM1. Not finished in class: finish before TP1.

Name:

## Part I. Names, numbers and text

### I.1 Print a result

In [ ]:
# A line starting with "#" is a comment. Python skips it.
print("Hello, PROJ951")

In [ ]:
size_kwp = 99           # a name on the left, a value on the right
building = "school"     # text goes between quotes
print(size_kwp)
print(f"The {building} roof takes {size_kwp} kWp")   # an f-string puts values inside a text

### I.2 Operations and units

`+`, `-`, `*`, `/` work as on a calculator. `**` gives a power: `1.05 ** 10` means 1.05 to the power 10.

In [ ]:
price_eur_per_wp = 1.10
capex = price_eur_per_wp * size_kwp * 1000    # the price holds per Wp, the size in kWp: times 1000
print(f"CAPEX: {capex:,.0f} EUR")             # ",.0f": thousands separator, no decimal
print(1.05 ** 10)

#### Exercise 1. A house

A 9 kWp system on a house costs 1.60 EUR/Wp (made-up value). Compute its investment in EUR.

In [ ]:
house_capex = ...  # TODO
print(f"House CAPEX: {house_capex:,.0f} EUR")

**Check.** 14,400 EUR. A value of 14.4 EUR means a missing factor 1000.

#### Exercise 2. One euro in ten years

The value today of an amount received in year $t$ is $\text{amount} / (1 + r)^t$. Compute the value today of 1,000 EUR received in year 10, at a rate of 5 %. Write the rate as `0.05`, never as `5`.

In [ ]:
r = 0.05
value_today = ...  # TODO
print(f"Value today: {value_today:.0f} EUR")

In [ ]:
assert abs(value_today - 613.9) < 0.1, "Check the power and the rate written as 0.05"
print("OK")

## Part II. Many values at once: numpy

`numpy` applies one operation to a whole list of values in one line. The course uses the name `np` for it.

In [ ]:
import numpy as np

years = np.arange(1, 26)      # the whole numbers from 1 to 25, the 26 stays out
print(years)

In [ ]:
growth = 1.02 ** (years - 1)  # a price rising 2 % per year, one line for the 25 years
print(growth.round(3))
print(f"Year 20: {growth[19]:.3f}")   # positions start at 0, so year 20 sits at position 19
print(f"Year 25: {growth[-1]:.3f}")   # [-1] reads the last value

#### Exercise 3. Discount factors

Compute the discount factor $1 / (1 + r)^t$ for each of the 25 years in one line, then their sum. The sum is the annuity factor of CM2.

In [ ]:
discount = ...  # TODO
annuity = ...  # TODO
print(f"Sum of the 25 discount factors: {annuity:.3f}")

In [ ]:
assert abs(annuity - 14.094) < 0.001, "Use the array years, not a single number"
print("OK")

### II.2 Two functions you meet again in TP1

- `np.where(condition, a, b)` gives `a` where the condition holds and `b` elsewhere.
- `np.minimum(x, y)` gives the smaller of two values, position by position.

In [ ]:
tariff = np.where(years <= 20, 0.10, 0.05)   # 0.10 during a 20-year contract, 0.05 after
print(tariff)

In [ ]:
production = np.array([0, 5, 12, 8, 1])   # kWh in five hours, made-up values
demand = np.array([3, 4, 6, 9, 2])        # kWh in the same five hours
print(np.minimum(production, demand))

#### Exercise 4. Production left over

In the five made-up hours, the building uses the smaller of production and demand. Compute the production left over in each hour, then its total.

In [ ]:
left_over = ...  # TODO
print(left_over, left_over.sum(), "kWh")

**Check.** 7 kWh in total, all in the second and third hours.

## Part III. Tables: pandas

`pandas` holds tables with named columns and, here, one row per hour. The course uses the name `pd` for it, and `plt` for the plots.

The next cell reads the case data, `proj951_case_data.csv`, from the course repository: one row per hour of 2019.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/Mohamed-ENNHIRI/PROJ951/main/data/proj951_case_data.csv"
df = pd.read_csv(url, index_col="time", parse_dates=True)
print(df.shape)     # (rows, columns): 8760 hours and 5 columns
df.head()           # the first five rows

In [ ]:
school = df["load_school_kwh"]     # one column, 8760 hourly values
print(f"School demand: {school.sum() / 1000:.1f} MWh per year")   # kWh to MWh: divide by 1000
print(f"Busiest hour:  {school.max():.1f} kWh, on {school.idxmax()}")

In [ ]:
name = "office"
print(df[f"load_{name}_kwh"].sum() / 1000)   # a column name built from a variable

#### Exercise 5. The three buildings

Compute the annual demand of the office and of the supermarket, in MWh.

In [ ]:
office_mwh = ...  # TODO
supermarket_mwh = ...  # TODO
print(f"Office:      {office_mwh:.0f} MWh")
print(f"Supermarket: {supermarket_mwh:.0f} MWh")

**Check.** 220 MWh and 650 MWh. A value near 220,000 means the division by 1000 is missing.

### III.2 Select dates and plot

`df.loc["start":"end"]` keeps the rows between two dates, both included.

In [ ]:
week = df.loc["2019-01-14":"2019-01-20"]     # Monday 14 to Sunday 20 January 2019
print(len(week), "hours")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(week["load_school_kwh"], label="School")
ax.set_ylabel("Energy per hour (kWh)")       # every axis carries a quantity and a unit
ax.set_title("Demand, 14 to 20 January 2019")
ax.legend()
plt.show()

#### Exercise 6. Two buildings on one plot

Complete the cell below: add the office demand of the same week, with the label `"Office"`. On which days does each building use the least?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(week["load_school_kwh"], label="School")
# TODO: your code here
ax.set_ylabel("Energy per hour (kWh)")
ax.set_title("Demand, 14 to 20 January 2019")
ax.legend()
plt.show()

**Answer.**

### III.3 Month by month

`resample("MS")` groups the hours by month. `.sum()` adds the hours of each month.

In [ ]:
monthly = school.resample("MS").sum() / 1000
monthly.index = monthly.index.strftime("%b")
monthly.plot(kind="bar", ylabel="School demand per month (MWh)")
plt.show()
monthly.round(1)

## Part IV. Running totals and conditions

`cumsum()` adds the values one after the other. A condition such as `cumulative >= 0` gives `True` or `False` for each row, and keeps the rows where the condition holds.

In [ ]:
cash = pd.Series([-100, 30, 30, 30, 30, 30], index=range(6))   # EUR, years 0 to 5, made-up values
table = pd.DataFrame({"cash": cash})
table["cumulative"] = table["cash"].cumsum()                   # a new column in the table
table["positive"] = table["cumulative"] >= 0
table

#### Exercise 7. First positive year

Find the first year when the cumulative total reaches zero or more. `table.index[condition]` keeps the years where the condition holds, and `.min()` takes the first one.

In [ ]:
first_year = ...  # TODO
print(f"First positive year: {first_year}")

**Check.** Year 4: −100 + 4 × 30 = +20 EUR.

## Part V. Errors seen often

Python stops at the first error and prints a message. Read the **last line** of the message first: it names the error.

| Last line of the message | Usual cause | Fix |
|---|---|---|
| `NameError: name 'df' is not defined` | A cell run before the cells above it | Runtime, Run all |
| `FileNotFoundError` or `HTTPError` | The CSV is missing, or no internet access | Check the connection, run the cell again |
| `KeyError: 'load_School_kwh'` | A column name with a typo or a capital letter | Copy the name from `df.head()` |
| `SyntaxError` | A missing quote, bracket or comma | Look at the line the arrow points to |
| A number 1000 times too big or too small | Wp and kWp, or kWh and MWh, mixed | Write the unit next to each value |

## What you carry to TP1

- The notebook copy in your Drive, every exercise done, every check `OK`.
- The seven patterns: `print(f"...")`, `**`, `np.arange`, `np.where`, `np.minimum`, `df["column"].sum()`, `cumsum()`.
- One question for the teacher, if a cell stayed unclear.